# Pipeline: 1168 Videos - Download + Extract + Train

**What:**
- Download Gillick 600 videos (pre-labeled)
- Improve labels for existing 481 YouTube videos
- Train on 1168 total videos

**Expected:** ~4-6 hours total
**Output:** F1 score on held-out comedians

In [ ]:
# @title Setup
!pip install -q librosa scikit-learn
import os
from google.colab import drive
drive.mount('/content/gdrive', force_auth_changed=True)
print('✓ Drive mounted')
print('GPU:', !nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo 'CPU')

In [ ]:
# @title 1. Find Available Gillick Videos
import subprocess

# Get Gillick IDs from Drive
with open('/content/gdrive/MyDrive/gillick_988_videos.txt') as f:
    all_ids = [l.strip() for l in f if l.strip()]

print(f'Total Gillick IDs: {len(all_ids)}')

# Test availability (first 100)
available = []
for vid in all_ids[:200]:
    result = subprocess.run(
        ['yt-dlp', '--dump-json', f'https://youtu.be/{vid}'],
        capture_output=True, timeout=30
    )
    if result.returncode == 0:
        available.append(vid)
        if len(available) % 50 == 0:
            print(f'Found {len(available)} available...')
    if len(available) >= 600:
        break

print(f'\nAvailable: {len(available)} / 200 tested')

# Save available list
with open('/tmp/gillick_available.txt', 'w') as f:
    f.write('\n'.join(available))
print(f'Saved to /tmp/gillick_available.txt')

In [ ]:
# @title 2. Download Gillick 600
from concurrent.futures import ThreadPoolExecutor
import subprocess
import os

os.makedirs('/content/gillick_audio', exist_ok=True)

def download_one(vid):
    out = f'/content/gillick_audio/{vid}.mp3'
    if os.path.exists(out):
        return vid, True
    r = subprocess.run([
        'yt-dlp', '--extract-audio', '--audio-format', 'mp3',
        '--audio-quality', '5', '-o', out,
        f'https://youtu.be/{vid}'
    ], capture_output=True, timeout=300)
    return vid, r.returncode == 0

print(f'Downloading {len(available)} videos...')
with ThreadPoolExecutor(max_workers=10) as ex:
    results = list(ex.map(download_one, available))

success = sum(1 for _, ok in results if ok)
print(f'Downloaded: {success}/{len(available)}')

In [ ]:
# @title 3. Extract Prosody for Downloaded Gillick
import numpy as np
import librosa
from tqdm import tqdm
import os

SR = 16000

def extract_prosody_23(y, sr):
    f = []
    try:
        f0, voiced, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0c = f0[~np.isnan(f0)]
        f.extend([np.mean(f0c), np.std(f0c), np.max(f0c),
                   np.min(f0c), np.sum(voiced)/len(voiced)])
    except:
        f.extend([0]*5)
    
    rms = librosa.feature.rms(y=y)[0]
    f.extend([np.mean(rms), np.std(rms), np.max(rms),
               np.min(rms), np.max(rms)-np.min(rms)])
    
    f.extend([len(y)/sr, len(y)/sr/(np.sum(rms>np.mean(rms))+1])
    
    try:
        sc = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
        sf = librosa.feature.spectral_flatness(y=y)[0]
        zc = librosa.feature.zero_crossing_rate(y)[0]
        f.extend([np.mean(sc), np.mean(sb), np.mean(sf),
                   np.mean(zc), np.std(zc)])
    except:
        f.extend([0]*5)
    
    try:
        hnr = librosa.effects.hpss(y)[1]
        hnr_val = np.mean(hnr)/(np.mean(np.abs(y))+1e-8)
    except:
        hnr_val = 0
    f.extend([hnr_val, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y))])
    
    try:
        d = np.diff(rms)
        f.extend([np.mean(np.abs(d)), np.max(np.abs(d))])
    except:
        f.extend([0]*2)
    
    return np.array(f, dtype=np.float32)

# Process downloaded files
gillick_dir = '/content/gillick_audio'
gillick_files = [f for f in os.listdir(gillick_dir) if f.endswith('.mp3')]
print(f'Found {len(gillick_files)} downloaded Gillick files')

gillick_data = []
for fname in tqdm(gillick_files):
    vid = fname.replace('.mp3', '')
    try:
        y, sr = librosa.load(f'{gillick_dir}/{fname}', sr=SR, mono=True)
        # Extract for full file ( Gillick has laughter timestamps as metadata)
        prosody = extract_prosody_23(y, sr)
        gillick_data.append({'vid': vid, 'prosody': prosody})
    except Exception as e:
        print(f'Error {vid}: {e}')

print(f'Extracted: {len(gillick_data)} files')

# Save
gillick_prosody = np.array([d['prosody'] for d in gillick_data])
gillick_vids = [d['vid'] for d in gillick_data]
np.savez_compressed('/content/gdrive/MyDrive/gillick_600_prosody.npz',
                    prosody=gillick_prosody, vids=gillick_vids)
print('Saved to Drive!')

In [ ]:
# @title 4. Improve Labels for Existing 481 YouTube
import numpy as np
from sklearn.linear_model import LogisticRegression

# Load original 87 (gold standard)
d87 = np.load('/content/gdrive/MyDrive/wavlm_training_data_expanded.npz',
              allow_pickle=True)
X_gold = d87['prosody'][:, :5]  # F0 features
y_gold = d87['labels']
print(f'Gold 87: {len(y_gold)} utts, {y_gold.sum()} pos')

# Train F0 model
print('Training F0 model...')
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_gold, y_gold)

# Load YouTube 481
d481 = np.load('/content/gdrive/MyDrive/FINAL_500plus_41feat.npz',
            allow_pickle=True)
X_yt = d481['features'][:, :5]
y_old = d481['labels']
print(f'YouTube 481: {len(y_old)} utts, {y_old.sum()} pos')

# Predict improved labels
y_new = model.predict(X_yt)
y_prob = model.predict_proba(X_yt)[:, 1]

print(f'Old: {y_old.sum()} pos, New: {y_new.sum()} pos')

# Save improved
np.savez_compressed('/content/gdrive/MyDrive/YouTube_481_improved.npz',
                    old_labels=y_old, new_labels=y_new,
                    probs=y_prob, vids=d481['vids'])
print('Saved improved labels!')

In [ ]:
# @title 5. Train Final Model on 1168 Videos
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

# Load all data
d87 = np.load('/content/gdrive/MyDrive/wavlm_training_data_expanded.npz', allow_pickle=True)
d481 = np.load('/content/gdrive/MyDrive/FINAL_500plus_41feat.npz', allow_pickle=True)
d_imp = np.load('/content/gdrive/MyDrive/YouTube_481_improved.npz', allow_pickle=True)

# Combine: 87 gold + 481 with improved labels
X1 = d87['prosody']  # (21468, 23)
y1 = d87['labels']
X2 = d481['features'][:, :23]  # (183087, 23)
y2 = d_imp['new_labels']  # Improved labels

X_all = np.vstack([X1, X2])
y_all = np.concatenate([y1, y2])
vids_all = ([str(u).rsplit('_',1)[0] for u in d87['uids']] + 
            [str(v) for v in d481['vids']])

print(f'Combined: {len(y_all)} utts, {y_all.sum()} pos ({100*y_all.mean():.1f}%)')

# Video-level split
from collections import defaultdict
vid_to_idx = defaultdict(list)
for i, v in enumerate(vids_all):
    vid_to_idx[v].append(i)

all_vids = list(vid_to_idx.keys())
np.random.seed(42)
np.random.shuffle(all_vids)
n_val = int(0.1 * len(all_vids))
n_test = int(0.1 * len(all_vids))

val_vids = set(all_vids[:n_val])
test_vids = set(all_vids[n_val:n_val+n_test])

train_idx = [i for v in all_vids[n_val+n_test:] for i in vid_to_idx[v]]
val_idx = [i for v in val_vids for i in vid_to_idx[v]]
test_idx = [i for v in test_vids for i in vid_to_idx[v]]

X_train, X_val, X_test = X_all[train_idx], X_all[val_idx], X_all[test_idx]
y_train, y_val, y_test = y_all[train_idx], y_all[val_idx], y_all[test_idx]

print(f'Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')

# Normalize
X_mean, X_std = X_train.mean(0), X_train.std(0) + 1e-8
X_train = (X_train - X_mean) / X_std
X_val = (X_val - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

# Dataset & DataLoader
class DS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(DS(X_train, y_train), batch_size=256, shuffle=True)
val_loader = DataLoader(DS(X_val, y_val), batch_size=256)
test_loader = DataLoader(DS(X_test, y_test), batch_size=256)

# Model
class MLP(torch.nn.Module):
    def __init__(self, d=23):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(d, 128), torch.nn.ReLU(), torch.nn.Dropout(0.3),
            torch.nn.Linear(128, 32), torch.nn.ReLU(),
            torch.nn.Linear(32, 1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP().to(device)
pos_w = torch.tensor([(len(y_train) - y_train.sum()) / max(1, y_train.sum())]).to(device)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_w)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

best_f1, best_state = 0, None
for epoch in range(30):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        criterion(model(x), y).backward()
        optimizer.step()
    
    model.eval()
    preds, labs = [], []
    with torch.no_grad():
        for x, y in val_loader:
            p = (torch.sigmoid(model(x.to(device))) > 0.5).cpu().numpy()
            preds.extend(p); labs.extend(y.numpy())
    
    f1 = f1_score(labs, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_state = model.state_dict().copy()
        torch.save(best_state, '/content/gdrive/MyDrive/best_prosody_mlp_1168.pt')
    
    print(f'Epoch {epoch+1}: Val F1 = {f1:.4f}')

# Final test
model.load_state_dict(best_state)
model.eval()
preds, labs = [], []
with torch.no_grad():
    for x, y in test_loader:
        p = (torch.sigmoid(model(x.to(device))) > 0.5).cpu().numpy()
        preds.extend(p); labs.extend(y.numpy())

test_f1 = f1_score(labs, preds)
print(f'\n=== TEST F1: {test_f1:.4f} ===')